# EUDR Coffee Risk Analytics — Phase 2

Phase 1 was Hansen loss on a map. Phase 2 makes it decision-grade:

| Added | Why |
|---|---|
| **JRC GFC2020 V3** forest baseline | The layer actually aligned with the EUDR cutoff (31 Dec 2020). Hansen alone answers a different question. |
| **RADD radar alerts** | Near-real-time, and radar sees through wet-season cloud over the Central Highlands. |
| **Legality overlay** | EUDR requires deforestation-free **and** legally produced. Satellites only cover the first half. |
| **Transparent risk score** | Ranks plots for field verification, explainable to an auditor and to the farmer. |
| **DDS-oriented export** | Geolocation + risk evidence in a bundle you can hand to the operator who files the statement. |

Cost is still $0: every dataset is an open Earth Engine asset.

> **Scope:** this produces evidence and a ranked worklist. It does not decide compliance and it does not file a DDS.

In [ ]:
# Run once (Colab). Skip if your local env already has these.
# !pip install -q earthengine-api geemap pandas

In [ ]:
import json, sys, os

import ee
import geemap

# Make the eudr_risk package importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))
from eudr_risk import alerts, config, dds, forest, pipeline, scoring

PROJECT = "YOUR_GEE_PROJECT"  # <-- your Earth Engine Cloud project id

# ee.Authenticate()   # first run on a new machine only
ee.Initialize(project=PROJECT)
print("Earth Engine ready. eudr_risk", __import__("eudr_risk").__version__)

## Step 0 — Verify the assets

Asset versions and band names drift between releases. Check them before you
trust a run. If the RADD date band is not called `Date`, set
`config.RADD_DATE_BAND` to whatever shows up here.

In [ ]:
print("JRC baseline :", config.JRC_GFC2020_ASSET)
print("Hansen       :", config.HANSEN_ASSET)
print("RADD         :", config.RADD_ASSET, "| geography:", config.RADD_GEOGRAPHY)
print("RADD bands   :", alerts.radd_band_names().getInfo())
print("JRC bands    :", ee.Image(config.JRC_GFC2020_ASSET).bandNames().getInfo())
print("Cutoff       :", config.CUTOFF_DATE, "| RADD YYDOY >=", config.CUTOFF_YYDOY)
print("Country risk :", config.COUNTRY_NAME, "=", config.COUNTRY_EUDR_RISK)

## Step 1 — Load plots

Swap `plots_sample.geojson` for your real boundaries. Every feature needs a
`plot_id`. Real farmer boundaries are personal data: keep them out of git
(the `.gitignore` already blocks `data/real/`).

In [ ]:
PLOTS_PATH = "../data/plots_sample.geojson"

with open(PLOTS_PATH, encoding="utf-8") as fh:
    plots_geojson = json.load(fh)

geometry_by_id = {
    f["properties"]["plot_id"]: f["geometry"] for f in plots_geojson["features"]
}
plots = geemap.geojson_to_ee(plots_geojson)
print("Loaded plots:", plots.size().getInfo())

## Step 2 — Extract features and score

`strict_jrc=True` requires post-cutoff loss to fall inside JRC 2020 forest,
which is the EUDR-faithful test. Set it `False` for a wider net that flags
more plots (useful for a first screening pass).

In [ ]:
features = pipeline.extract_features(plots, strict_jrc=True)
table = scoring.score_dataframe(pipeline.to_dataframe(features))
summary = scoring.summarise(table)

print(json.dumps(summary, indent=2))
table

## Step 3 — Map

Green plots are clean, orange need review, red are high risk. Red pixels are
post-cutoff tree-cover loss; magenta are recent radar alerts.

In [ ]:
loss = forest.loss_after_cutoff_on_jrc_forest()
radd = alerts.radd_alerts_after_cutoff()

Map = geemap.Map()
Map.centerObject(plots, 14)
Map.add_basemap("SATELLITE")

Map.addLayer(
    forest.jrc_forest_2020().selfMask(),
    {"palette": ["#4d9221"]},
    "JRC forest 2020",
    False,
)
Map.addLayer(loss.selfMask(), {"palette": ["red"]}, "Loss after cutoff")
Map.addLayer(radd.selfMask(), {"palette": ["magenta"]}, "RADD alerts 2021+")

tier_colors = {"low": "00FF00", "standard": "FFA500", "high": "FF0000"}
for tier, color in tier_colors.items():
    ids = table.loc[table["risk_tier"] == tier, "plot_id"].tolist()
    if not ids:
        continue
    subset = features.filter(ee.Filter.inList("plot_id", ids))
    Map.addLayer(
        subset.style(color=color, fillColor=color + "33", width=2),
        {},
        f"Plots - {tier}",
    )

Map

## Step 4 — Export the due-diligence bundle

Four artefacts: the risk table, a DDS-oriented GeoJSON, a summary, and a
printable field checklist ordered by risk. The checklist is the point of the
whole exercise: a field team gets a short ranked list instead of every farm.

In [ ]:
OUT_DIR = "../out"

doc = dds.build_dds(
    table,
    geometry_by_id,
    operator={"name": "<OPERATOR LEGAL NAME>", "eori": "<EORI>"},
    harvest_year=2026,
)
paths = dds.write_outputs(table, doc, summary, OUT_DIR)
paths["checklist"] = dds.write_field_checklist(table, OUT_DIR)

for key, path in paths.items():
    print(f"{key:10s} {path}")

## Phase 3 — what is still open

- **Vietnam legality layer.** Load the official 3-loai-rung zoning (rung dac
  dung / phong ho / san xuat) as an EE asset and pass it via
  `pipeline.extract_features(..., legality_fc=...)`. WDPA is only a proxy.
- **Join to the national EUDR cultivation-zone database** so each plot carries
  its official traceability code alongside our risk fields.
- **Chain of custody.** Plot-level risk is useless if verified and
  non-verified lots blend at the collector. Model the lot, not just the plot.
- **Validate the score.** Ground-truth a sample of flagged plots, then tune
  `config.DEFAULT_WEIGHTS` against what the field team actually found.
- **Scheduled runs.** Re-run monthly as new RADD alerts land; alert on change
  rather than re-reading the whole portfolio.